In [1]:
import sys
from pathlib import Path

ROOT = str(Path.cwd().parent)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split as tts
from sklearn.preprocessing import MinMaxScaler
import optuna
from optuna.trial import TrialState

from src.data_loader import load_all_datasets
from src.preprocessing import add_rul_column, add_binary_label, drop_useless_columns, normalize_by_operating_condition
from src.feature_engineering import build_features
from src.models.deep_learning import DEVICE
from src.models.train import create_sequences
from src.models.evaluate import regression_metrics
from src.config import DATASETS, PROCESSED_DIR, MODELS_DIR, SEQUENCE_LENGTH, RANDOM_STATE, WINDOW_SIZE
from src.utils import set_seed

set_seed()
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Ti


In [2]:
with open(PROCESSED_DIR / "metadata.json") as f:
    meta = json.load(f)

feature_cols = meta["feature_cols"]

processed = {}
for ds_id in DATASETS:
    train = pd.read_parquet(PROCESSED_DIR / f"train_{ds_id}.parquet")
    test = pd.read_parquet(PROCESSED_DIR / f"test_{ds_id}.parquet")
    processed[ds_id] = {"train": train, "test": test}

ds_id = "FD001"
train_raw = processed[ds_id]["train"]

exclude = {"unit_id", "cycle", "rul", "label"}
cols = [c for c in train_raw.columns if c not in exclude]

train_norm = normalize_by_operating_condition(train_raw)

units = train_norm["unit_id"].unique()
train_ids, val_ids = tts(units, test_size=0.2, random_state=42)

train_split = train_norm[train_norm["unit_id"].isin(train_ids)].copy()
val_split = train_norm[train_norm["unit_id"].isin(val_ids)].copy()

feature_scaler = MinMaxScaler()
train_split[cols] = feature_scaler.fit_transform(train_split[cols])
val_split[cols] = feature_scaler.transform(val_split[cols])

rul_max = 125.0

X_train, y_train_raw = create_sequences(train_split, cols, "rul", SEQUENCE_LENGTH)
y_train = y_train_raw / rul_max

X_val, y_val_raw = create_sequences(val_split, cols, "rul", SEQUENCE_LENGTH)
y_val = y_val_raw / rul_max

n_features = X_train.shape[2]
print(f"Features: {n_features} | Train: {X_train.shape} | Val: {X_val.shape}")

Features: 58 | Train: (10241, 80, 58) | Val: (2490, 80, 58)


In [3]:
class FlexibleModel(nn.Module):
    def __init__(self, n_features, model_type, hidden_size, num_layers, dropout, d_model=64, nhead=4):
        super().__init__()
        self.model_type = model_type

        if model_type == "lstm":
            self.rnn = nn.LSTM(n_features, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        elif model_type == "gru":
            self.rnn = nn.GRU(n_features, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        elif model_type == "tcn":
            layers = []
            in_ch = n_features
            for dilation in [1, 2, 4, 8, 16]:
                padding = (3 - 1) * dilation
                layers.append(nn.Conv1d(in_ch, hidden_size, 3, padding=padding, dilation=dilation))
                layers.append(nn.BatchNorm1d(hidden_size))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout))
                in_ch = hidden_size
            self.tcn = nn.Sequential(*layers)
        elif model_type == "transformer":
            self.input_proj = nn.Linear(n_features, d_model)
            self.pos_enc = nn.Parameter(torch.randn(1, 200, d_model) * 0.02)
            encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=hidden_size, dropout=dropout, batch_first=True)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        out_size = d_model if model_type == "transformer" else hidden_size
        self.bn = nn.BatchNorm1d(out_size)
        self.fc1 = nn.Linear(out_size, 48)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(48, 1)

    def forward(self, x):
        if self.model_type in ["lstm", "gru"]:
            out, _ = self.rnn(x)
            out = out[:, -1, :]
        elif self.model_type == "tcn":
            out = self.tcn(x.permute(0, 2, 1))
            out = out[:, :, :x.size(1)].mean(dim=2)
        elif self.model_type == "transformer":
            out = self.input_proj(x) + self.pos_enc[:, :x.size(1), :]
            out = self.transformer(out).mean(dim=1)

        out = self.bn(out)
        out = self.drop(self.relu(self.fc1(out)))
        return self.fc2(out).squeeze(-1)

In [4]:
def train_and_evaluate(trial, model_type):
    hidden_size = trial.suggest_int("hidden_size", 64, 256, step=64)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.2, 0.5, step=0.05)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    if model_type == "transformer":
        d_model = trial.suggest_categorical("d_model", [32, 64, 128])
        nhead = trial.suggest_categorical("nhead", [2, 4])
        if d_model % nhead != 0:
            nhead = 2
        model = FlexibleModel(n_features, model_type, hidden_size, num_layers, dropout, d_model, nhead).to(DEVICE)
    else:
        model = FlexibleModel(n_features, model_type, hidden_size, num_layers, dropout).to(DEVICE)

    X_tr_t = torch.FloatTensor(X_train).to(DEVICE)
    y_tr_t = torch.FloatTensor(y_train).to(DEVICE)
    X_val_t = torch.FloatTensor(X_val).to(DEVICE)
    y_val_t = torch.FloatTensor(y_val).to(DEVICE)

    train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True, drop_last=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5, min_lr=1e-6)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(80):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t).item()

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 10:
            break

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    model.eval()
    with torch.no_grad():
        y_pred = model(X_val_t).cpu().numpy() * rul_max
    y_real = y_val * rul_max
    rmse = np.sqrt(np.mean((y_pred - y_real) ** 2))

    return rmse

In [5]:
studies = {}

for ds_id in DATASETS:
    train_raw = processed[ds_id]["train"]
    exclude_cols = {"unit_id", "cycle", "rul", "label"}
    ds_cols = [c for c in train_raw.columns if c not in exclude_cols]

    tr_norm = normalize_by_operating_condition(train_raw)
    units = tr_norm["unit_id"].unique()
    tr_ids, val_ids = tts(units, test_size=0.2, random_state=42)

    tr_split = tr_norm[tr_norm["unit_id"].isin(tr_ids)].copy()
    vl_split = tr_norm[tr_norm["unit_id"].isin(val_ids)].copy()

    scaler = MinMaxScaler()
    tr_split[ds_cols] = scaler.fit_transform(tr_split[ds_cols])
    vl_split[ds_cols] = scaler.transform(vl_split[ds_cols])

    X_tr, y_tr_raw = create_sequences(tr_split, ds_cols, "rul", SEQUENCE_LENGTH)
    y_tr = y_tr_raw / rul_max
    X_vl, y_vl_raw = create_sequences(vl_split, ds_cols, "rul", SEQUENCE_LENGTH)
    y_vl = y_vl_raw / rul_max
    n_feat = X_tr.shape[2]

    X_tr_t = torch.FloatTensor(X_tr).to(DEVICE)
    y_tr_t = torch.FloatTensor(y_tr).to(DEVICE)
    X_vl_t = torch.FloatTensor(X_vl).to(DEVICE)
    y_vl_t = torch.FloatTensor(y_vl).to(DEVICE)

    for model_type in ["lstm", "gru", "tcn", "transformer"]:
        key = f"{model_type}_{ds_id}"
        print(f"\n{'='*50}")
        print(f"  Optuna: {model_type.upper()} × {ds_id}")
        print(f"{'='*50}")

        def objective(trial, mt=model_type, nf=n_feat):
            set_seed()
            hidden_size = trial.suggest_int("hidden_size", 64, 256, step=64)
            num_layers = trial.suggest_int("num_layers", 1, 3)
            dropout = trial.suggest_float("dropout", 0.2, 0.5, step=0.05)
            lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
            weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
            batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

            if mt == "transformer":
                d_model = trial.suggest_categorical("d_model", [32, 64, 128])
                nhead = trial.suggest_categorical("nhead", [2, 4])
                if d_model % nhead != 0:
                    nhead = 2
                model = FlexibleModel(nf, mt, hidden_size, num_layers, dropout, d_model, nhead).to(DEVICE)
            else:
                model = FlexibleModel(nf, mt, hidden_size, num_layers, dropout).to(DEVICE)

            train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True, drop_last=True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5, min_lr=1e-6)
            criterion = nn.MSELoss()

            best_val_loss = float("inf")
            patience_counter = 0

            for epoch in range(80):
                model.train()
                for X_batch, y_batch in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(X_batch), y_batch)
                    loss.backward()
                    optimizer.step()

                model.eval()
                with torch.no_grad():
                    val_loss = criterion(model(X_vl_t), y_vl_t).item()

                scheduler.step(val_loss)

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1

                if patience_counter >= 10:
                    break

                trial.report(val_loss, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

            model.eval()
            with torch.no_grad():
                y_pred = model(X_vl_t).cpu().numpy() * rul_max
            y_real = y_vl * rul_max
            rmse = np.sqrt(np.mean((y_pred - y_real) ** 2))
            return rmse

        study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
        study.optimize(objective, n_trials=50, show_progress_bar=True)
        studies[key] = study
        print(f"  Mejor {model_type.upper()} × {ds_id}: RMSE={study.best_value:.2f}")
        print(f"  Params: {study.best_params}")

[I 2026-04-14 01:50:00,461] A new study created in memory with name: no-name-5d987042-34fa-44c5-bf37-78051f3fa76a



  Optuna: LSTM × FD001


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-14 01:50:23,579] Trial 0 finished with value: 17.31903722887492 and parameters: {'hidden_size': 256, 'num_layers': 3, 'dropout': 0.45, 'lr': 0.00015471236538491475, 'weight_decay': 1.2478468116288174e-05, 'batch_size': 128}. Best is trial 0 with value: 17.31903722887492.
[I 2026-04-14 01:50:40,999] Trial 1 finished with value: 21.753235295600867 and parameters: {'hidden_size': 256, 'num_layers': 1, 'dropout': 0.4, 'lr': 0.00014279616146702103, 'weight_decay': 9.205157705005388e-05, 'batch_size': 64}. Best is trial 0 with value: 17.31903722887492.
[I 2026-04-14 01:50:59,984] Trial 2 finished with value: 12.920263481113501 and parameters: {'hidden_size': 64, 'num_layers': 3, 'dropout': 0.5, 'lr': 0.0002033244920460197, 'weight_decay': 0.0002669903343401956, 'batch_size': 32}. Best is trial 2 with value: 12.920263481113501.
[I 2026-04-14 01:51:53,771] Trial 3 finished with value: 16.989769137270056 and parameters: {'hidden_size': 192, 'num_layers': 1, 'dropout': 0.2, 'lr': 0.00

[I 2026-04-14 01:56:26,770] A new study created in memory with name: no-name-c9e3dfd1-ae19-4b1c-bde2-31c86a89d4b0


...

[I 2026-04-14 09:32:30,748] Trial 0 finished with value: 26.337223169014926 and parameters: {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.5, 'lr': 0.004098941150156556, 'weight_decay': 3.9482141169476654e-05, 'batch_size': 32}. Best is trial 0 with value: 26.337223169014926.
[I 2026-04-14 09:32:50,005] Trial 1 finished with value: 25.080036226407593 and parameters: {'hidden_size': 192, 'num_layers': 3, 'dropout': 0.4, 'lr': 0.000652168780261719, 'weight_decay': 4.9928525530756865e-05, 'batch_size': 64}. Best is trial 1 with value: 25.080036226407593.
[I 2026-04-14 09:33:26,548] Trial 2 finished with value: 19.667657110153808 and parameters: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.25, 'lr': 0.00011041804488476341, 'weight_decay': 0.0008747461708385426, 'batch_size': 64}. Best is trial 2 with value: 19.667657110153808.
[I 2026-04-14 09:33:44,543] Trial 3 finished with value: 20.482242059615626 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.3000000000000

[I 2026-04-14 09:39:15,053] A new study created in memory with name: no-name-a05fa999-9cd9-46ed-8cce-a28f1e34193c


[I 2026-04-14 09:39:15,051] Trial 49 pruned. 
  Mejor TCN × FD004: RMSE=19.67
  Params: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.25, 'lr': 0.00011041804488476341, 'weight_decay': 0.0008747461708385426, 'batch_size': 64}

  Optuna: TRANSFORMER × FD004


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-14 09:39:40,716] Trial 0 finished with value: 22.35622522120325 and parameters: {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.0005130239448575537, 'weight_decay': 3.478565724090299e-05, 'batch_size': 64, 'd_model': 128, 'nhead': 2}. Best is trial 0 with value: 22.35622522120325.
[I 2026-04-14 09:39:57,117] Trial 1 finished with value: 19.385870213296286 and parameters: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.30000000000000004, 'lr': 0.0007620668324728923, 'weight_decay': 1.690566336760072e-05, 'batch_size': 128, 'd_model': 32, 'nhead': 2}. Best is trial 1 with value: 19.385870213296286.
[I 2026-04-14 09:40:34,902] Trial 2 finished with value: 17.03587307581946 and parameters: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.00015301365535102992, 'weight_decay': 2.067574233986724e-05, 'batch_size': 64, 'd_model': 32, 'nhead': 4}. Best is trial 2 with value: 17.03587307581946.
[I 2026-04-14 09:41:00,229] Trial 3 finished with value: 19.37

In [6]:
print("=" * 60)
print("  MEJORES HIPERPARÁMETROS (OPTUNA)")
print("=" * 60)
for key, study in studies.items():
    print(f"\n  {key.upper()} — RMSE: {study.best_value:.2f}")
    for k, v in study.best_params.items():
        print(f"    {k}: {v}")
print("=" * 60)

  MEJORES HIPERPARÁMETROS (OPTUNA)

  LSTM_FD001 — RMSE: 12.54
    hidden_size: 192
    num_layers: 3
    dropout: 0.25
    lr: 0.00017923384499941508
    weight_decay: 0.00044228119636054234
    batch_size: 32

  GRU_FD001 — RMSE: 12.22
    hidden_size: 128
    num_layers: 3
    dropout: 0.4
    lr: 0.0044106771579938715
    weight_decay: 7.579756671974912e-05
    batch_size: 32

  TCN_FD001 — RMSE: 11.65
    hidden_size: 192
    num_layers: 3
    dropout: 0.30000000000000004
    lr: 0.00025963011601906063
    weight_decay: 7.681313671833259e-05
    batch_size: 128

  TRANSFORMER_FD001 — RMSE: 11.84
    hidden_size: 192
    num_layers: 1
    dropout: 0.30000000000000004
    lr: 0.0002898239716775429
    weight_decay: 0.0001595655327096281
    batch_size: 64
    d_model: 128
    nhead: 2

  LSTM_FD002 — RMSE: 10.76
    hidden_size: 64
    num_layers: 1
    dropout: 0.2
    lr: 0.002944227020060717
    weight_decay: 0.00018379666359595058
    batch_size: 32

  GRU_FD002 — RMSE: 11.24
  

In [7]:
from src.models.train import train_dl_model, predict_dl

all_best_results = {}
all_best_models = {}

for ds_id in DATASETS:
    print(f"\n{'='*50}")
    print(f"  Reentrenando {ds_id} con mejores params")
    print(f"{'='*50}")

    train_raw = processed[ds_id]["train"]
    exclude_cols = {"unit_id", "cycle", "rul", "label"}
    ds_cols = [c for c in train_raw.columns if c not in exclude_cols]

    tr_norm = normalize_by_operating_condition(train_raw)
    units = tr_norm["unit_id"].unique()
    tr_ids, val_ids = tts(units, test_size=0.2, random_state=42)

    tr_split = tr_norm[tr_norm["unit_id"].isin(tr_ids)].copy()
    vl_split = tr_norm[tr_norm["unit_id"].isin(val_ids)].copy()

    scaler = MinMaxScaler()
    tr_split[ds_cols] = scaler.fit_transform(tr_split[ds_cols])
    vl_split[ds_cols] = scaler.transform(vl_split[ds_cols])

    X_tr, y_tr_raw = create_sequences(tr_split, ds_cols, "rul", SEQUENCE_LENGTH)
    y_tr = y_tr_raw / rul_max
    X_vl, y_vl_raw = create_sequences(vl_split, ds_cols, "rul", SEQUENCE_LENGTH)
    y_vl = y_vl_raw / rul_max
    n_feat = X_tr.shape[2]

    ds_models = {}
    ds_results = {}

    for model_type in ["lstm", "gru", "tcn", "transformer"]:
        key = f"{model_type}_{ds_id}"
        p = studies[key].best_params
        set_seed()

        if model_type == "transformer":
            d_model = p.get("d_model", 64)
            nhead = p.get("nhead", 4)
            model = FlexibleModel(n_feat, model_type, p["hidden_size"], p["num_layers"], p["dropout"], d_model, nhead).to(DEVICE)
        else:
            model = FlexibleModel(n_feat, model_type, p["hidden_size"], p["num_layers"], p["dropout"]).to(DEVICE)

        history = train_dl_model(model, X_tr, y_tr, f"{model_type}_optuna", ds_id)

        y_pred = predict_dl(model, X_vl) * rul_max
        y_real = y_vl * rul_max
        metrics = regression_metrics(y_real, y_pred)
        ds_models[model_type] = model
        ds_results[model_type] = metrics
        print(f"  {model_type}: RMSE={metrics['rmse']:.2f} | R²={metrics['r2']:.4f}")

    # Ensemble
    preds = [predict_dl(m, X_vl) * rul_max for m in ds_models.values()]
    ensemble_pred = np.mean(preds, axis=0)
    ensemble_metrics = regression_metrics(y_real, ensemble_pred)
    ds_results["ensemble"] = ensemble_metrics
    print(f"  ENSEMBLE: RMSE={ensemble_metrics['rmse']:.2f} | R²={ensemble_metrics['r2']:.4f}")

    all_best_results[ds_id] = ds_results
    all_best_models[ds_id] = ds_models


  Reentrenando FD001 con mejores params
  Epoch 10/100 — loss: 0.0118 — val_loss: 0.8474 — val_mae: 0.8423
  Early stopping en epoch 13
✓ lstm_optuna guardado en C:\Users\OEM\Desktop\Proyectos personales\NASA_C-MAPSS_Data\models\lstm_optuna_FD001.pt
  lstm: RMSE=14.24 | R²=0.8730
  Epoch 10/100 — loss: 0.0143 — val_loss: 0.0198 — val_mae: 0.1061
  Epoch 20/100 — loss: 0.0098 — val_loss: 0.0157 — val_mae: 0.1029
  Epoch 30/100 — loss: 0.0085 — val_loss: 0.0032 — val_mae: 0.0436
  Epoch 40/100 — loss: 0.0069 — val_loss: 0.0023 — val_mae: 0.0383
  Epoch 50/100 — loss: 0.0058 — val_loss: 0.0033 — val_mae: 0.0438
  Epoch 60/100 — loss: 0.0053 — val_loss: 0.0014 — val_mae: 0.0289
  Epoch 70/100 — loss: 0.0050 — val_loss: 0.0013 — val_mae: 0.0290
  Epoch 80/100 — loss: 0.0048 — val_loss: 0.0015 — val_mae: 0.0300
  Epoch 90/100 — loss: 0.0049 — val_loss: 0.0011 — val_mae: 0.0254
  Epoch 100/100 — loss: 0.0047 — val_loss: 0.0009 — val_mae: 0.0233
✓ gru_optuna guardado en C:\Users\OEM\Desktop\P

In [8]:
print("=" * 60)
print("  RESULTADOS FINALES CON OPTUNA + ENSEMBLE")
print("=" * 60)
for ds_id in DATASETS:
    print(f"\n  {ds_id}:")
    for name, m in all_best_results[ds_id].items():
        marker = " ★" if name == "ensemble" else ""
        print(f"    {name}: RMSE={m['rmse']:.2f} | R²={m['r2']:.4f}{marker}")
print("=" * 60)

  RESULTADOS FINALES CON OPTUNA + ENSEMBLE

  FD001:
    lstm: RMSE=14.24 | R²=0.8730
    gru: RMSE=14.89 | R²=0.8610
    tcn: RMSE=14.39 | R²=0.8703
    transformer: RMSE=13.16 | R²=0.8915
    ensemble: RMSE=11.97 | R²=0.9102 ★

  FD002:
    lstm: RMSE=12.80 | R²=0.8920
    gru: RMSE=14.86 | R²=0.8546
    tcn: RMSE=13.43 | R²=0.8812
    transformer: RMSE=14.28 | R²=0.8657
    ensemble: RMSE=11.95 | R²=0.9060 ★

  FD003:
    lstm: RMSE=15.56 | R²=0.8599
    gru: RMSE=13.71 | R²=0.8911
    tcn: RMSE=16.82 | R²=0.8362
    transformer: RMSE=13.34 | R²=0.8970
    ensemble: RMSE=13.46 | R²=0.8951 ★

  FD004:
    lstm: RMSE=17.92 | R²=0.8236
    gru: RMSE=18.66 | R²=0.8089
    tcn: RMSE=21.48 | R²=0.7466
    transformer: RMSE=17.77 | R²=0.8266
    ensemble: RMSE=17.32 | R²=0.8354 ★


In [9]:
import json

save_data = {}
for key, study in studies.items():
    save_data[key] = {"best_rmse": study.best_value, "best_params": study.best_params}

for ds_id in DATASETS:
    for name, m in all_best_results[ds_id].items():
        save_data[f"result_{name}_{ds_id}"] = m

with open(PROCESSED_DIR / "optuna_results.json", "w") as f:
    json.dump(save_data, f, indent=2, default=str)

print("✓ Resultados guardados")

✓ Resultados guardados
